# Práctica 06 — Análisis de sentimiento clásico

**Curso:** NLP Clásico con Python · **Clase 06** · **Tiempo estimado:** 60-75 minutos
**Notebook de clase:** `notebooks_clase/clase06_sentimiento_clasico.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una o varias celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Los ejercicios 1 a 4 usan un **léxico y unas frases minúsculas** (celda de datos tras el setup) para que puedas calcular todo a mano; el ejercicio 5 y el reto usan las reseñas y el léxico de `datos/` (en Colab, sube la carpeta `datos/`). Son textos sintéticos escritos para el curso (*ejemplo ilustrativo*).
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. Librerías: `numpy`, `pandas`, `scipy` y `scikit-learn`.

---
## Celda 0 — Setup y datos

In [ ]:
import os
import re
import unicodedata

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

IDIOMAS = ("spa", "ita")
ARCHIVO_RESENAS = {"spa": "resenas_es.csv", "ita": "resenas_it.csv"}
ARCHIVO_LEXICO = {"spa": "lexico_polaridad_es.csv", "ita": "lexico_polaridad_it.csv"}
CLASES = ["negativo", "neutro", "positivo"]

def normalizar(texto):
    """Minúsculas y sin acentos: así el léxico también funciona con el texto bruto que perdió los acentos."""
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

def tokenizar(texto):
    """Palabras y signos de puntuación (los signos delimitan la ventana de la negación)."""
    return re.findall(r"\w+|[.,;:!?()]", normalizar(texto))

def cargar_lexico(idioma):
    tabla = pd.read_csv(cargar_datos(ARCHIVO_LEXICO[idioma]))
    return {normalizar(p): float(v) for p, v in zip(tabla["palabra"], tabla["polaridad"])}

def analizar(texto, lexico, idioma, ventana=3, negacion=True, intensificadores=True):
    """Devuelve (puntuación total, lista de aportes (palabra, valor base, valor final))."""
    tokens = tokenizar(texto)
    total, aportes = 0.0, []
    for i, palabra in enumerate(tokens):
        if palabra not in lexico:
            continue
        valor = lexico[palabra]
        multiplicador, invertir = 1.0, False
        for j in range(i - 1, max(i - ventana, 0) - 1, -1):    # hacia atrás, hasta `ventana` tokens
            anterior = tokens[j]
            if not anterior[0].isalnum() or anterior in CORTES[idioma]:
                break                                          # puntuación o «pero»: fin de la cláusula
            if negacion and anterior in NEGADORES[idioma]:
                invertir = True                                # varios negadores seguidos invierten UNA sola vez
            elif intensificadores and anterior in INTENSIFICADORES[idioma]:
                multiplicador *= INTENSIFICADORES[idioma][anterior]
        final = valor * multiplicador * (-1 if invertir else 1)
        aportes.append((palabra, valor, final))
        total += final
    return total, aportes

def puntuar(texto, lexico, idioma, **opciones):
    return analizar(texto, lexico, idioma, **opciones)[0]


print("Librerías cargadas correctamente.")

In [ ]:
# datos-es
LEXICO_MINI = {"spa": {"bueno": 1, "excelente": 2, "rápido": 1, "malo": -1, "horrible": -2, "lento": -1, "caro": -1},
               "ita": {"buono": 1, "eccellente": 2, "veloce": 1, "cattivo": -1, "orribile": -2, "lento": -1, "caro": -1}}
NEGADORES_P = {"spa": {"no", "nunca", "sin"}, "ita": {"non", "mai", "senza"}}
FRASES_P = {
    "spa": ["el servicio es bueno y rápido",
            "la comida es excelente",
            "el envío es lento y caro",
            "no es bueno",
            "nunca es lento",
            "el trato fue horrible",
            "el envío llegó ayer"],
    "ita": ["il servizio è buono e veloce",
            "il cibo è eccellente",
            "l'ordine è lento e caro",
            "non è buono",
            "mai lento",
            "il trattamento è stato orribile",
            "l'ordine è arrivato ieri"],
}
NEGADORES = {"spa": {"no", "ni", "nunca", "jamas", "tampoco", "sin", "nada", "ningun", "ninguno", "ninguna"},
             "ita": {"non", "mai", "senza", "nessun", "nessuno", "nessuna", "nemmeno", "neppure", "niente", "nulla"}}
INTENSIFICADORES = {"spa": {"muy": 1.5, "super": 1.5, "tremendamente": 1.5, "realmente": 1.5, "totalmente": 1.5,
                            "bastante": 1.25, "demasiado": 1.25, "poco": 0.5, "algo": 0.5},
                    "ita": {"molto": 1.5, "super": 1.5, "davvero": 1.5, "veramente": 1.5, "assolutamente": 1.5,
                            "estremamente": 1.5, "troppo": 1.25, "abbastanza": 1.25, "poco": 0.5, "alquanto": 0.75}}
CORTES = {"spa": {"pero", "aunque", "sino"}, "ita": {"ma", "pero", "sebbene", "tuttavia"}}   # las adversativas cortan la ventana

## Ejercicio 1 — Un analizador léxico mínimo

Con `LEXICO_MINI` (un diccionario `palabra → valor`), completa `puntuar_simple(frase, lexico)`: tokeniza con `\w+` en minúsculas y **suma el valor de cada palabra que esté en el léxico** (las que no están valen 0).

Antes de programar, calcula a mano las puntuaciones de las siete frases de `FRASES_P["spa"]`: `2, 2, −2, 1, −1, −2, 0`. Fíjate en que «no es bueno» **da +1**: todavía no tratamos la negación.

In [ ]:
def puntuar_simple(frase, lexico):
    tokens = re.findall(r"\w+", frase.lower())
    total = None        # TODO: suma de los valores de las palabras que están en el léxico
    return total

puntos_1 = {idioma: [puntuar_simple(f, LEXICO_MINI[idioma]) for f in FRASES_P[idioma]] for idioma in IDIOMAS}

# Comprobación automática
assert all(p is not None for idioma in IDIOMAS for p in puntos_1[idioma]), "Completa las líneas marcadas con TODO"
for idioma in IDIOMAS:
    assert puntos_1[idioma] == [2, 2, -2, 1, -1, -2, 0], "las puntuaciones deben ser 2, 2, −2, 1, −1, −2, 0"
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — Negación con ventana

Completa `puntuar_neg(frase, lexico, negadores, ventana)`: igual que antes, pero si **alguno de los `ventana` tokens anteriores** a una palabra del léxico es un negador, **invierte su signo** (una sola vez aunque haya varios negadores).

Calcula a mano: con `ventana=2`, «no es bueno» vale −1 y «nunca es lento» vale +1. Y «no es un servicio bueno» vale **+1** con `ventana=2` (el `no` queda a cuatro tokens de `bueno`) pero **−1** con `ventana=4`: la ventana tiene un precio.

In [ ]:
def puntuar_neg(frase, lexico, negadores, ventana):
    tokens = re.findall(r"\w+", frase.lower())
    total = 0
    for i, t in enumerate(tokens):
        if t in lexico:
            valor = lexico[t]
            anteriores = tokens[max(0, i - ventana):i]
            # TODO: si algún token de `anteriores` está en `negadores`, invierte el signo de `valor`
            total += valor
    return total

FRASE_LEJOS = {"spa": "no es un servicio bueno", "ita": "non è un servizio buono"}   # dato-es
res_2 = {idioma: ([puntuar_neg(f, LEXICO_MINI[idioma], NEGADORES_P[idioma], 2) for f in FRASES_P[idioma]],
                  puntuar_neg(FRASE_LEJOS[idioma], LEXICO_MINI[idioma], NEGADORES_P[idioma], 2),
                  puntuar_neg(FRASE_LEJOS[idioma], LEXICO_MINI[idioma], NEGADORES_P[idioma], 4)) for idioma in IDIOMAS}

# Comprobación automática
for idioma in IDIOMAS:
    lista, cerca, lejos = res_2[idioma]
    assert lista == [2, 2, -2, -1, 1, -2, 0], "con ventana=2 las puntuaciones deben ser 2, 2, −2, −1, 1, −2, 0"
    assert cerca == 1 and lejos == -1, "con ventana=2 la frase lejana vale +1 y con ventana=4 vale −1"
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — De la puntuación a la clase, exactitud y umbral

Completa tres funciones:

- `clasificar(p, umbral)`: `"positivo"` si `p > umbral`, `"negativo"` si `p < -umbral`, `"neutro"` en otro caso;
- `exactitud(y, pred)`: fracción de aciertos;
- `elegir_umbral(puntos, y, candidatos)`: el umbral de `candidatos` con mayor exactitud (a igualdad, el **menor**).

Con las ocho puntuaciones de `PUNTOS` y sus etiquetas reales `ETIQUETAS`, calcula a mano la exactitud con `umbral=0` (**6 de 8 = 0,75**) y con `umbral=1` (también 6 de 8) y con `umbral=2` (4 de 8).

In [ ]:
PUNTOS = [2, -1, 0, 1, -2, 0, 3, -1]
ETIQUETAS = ["positivo", "negativo", "neutro", "neutro", "negativo", "neutro", "positivo", "positivo"]

def clasificar(p, umbral):
    # TODO: devuelve "positivo", "negativo" o "neutro"
    return None

def exactitud(y, pred):
    # TODO: fracción de posiciones en que y y pred coinciden
    return None

def elegir_umbral(puntos, y, candidatos):
    # TODO: prueba cada candidato y devuelve el de mayor exactitud (el menor si hay empate)
    return None

# Comprobación automática
assert clasificar(0, 0) is not None and exactitud(["a"], ["a"]) is not None and elegir_umbral(PUNTOS, ETIQUETAS, (0, 1)) is not None, "Completa las líneas marcadas con TODO"
assert [clasificar(p, 0) for p in (2, -1, 0)] == ["positivo", "negativo", "neutro"], "con umbral 0: 2 → positivo, −1 → negativo, 0 → neutro"
assert clasificar(1, 1) == "neutro" and clasificar(-1, 1) == "neutro", "el umbral es estricto: p = umbral es neutro"
assert exactitud(ETIQUETAS, [clasificar(p, 0) for p in PUNTOS]) == 0.75
assert exactitud(ETIQUETAS, [clasificar(p, 1) for p in PUNTOS]) == 0.75
assert exactitud(ETIQUETAS, [clasificar(p, 2) for p in PUNTOS]) == 0.5
assert elegir_umbral(PUNTOS, ETIQUETAS, (2, 1, 0)) == 0, "0 y 1 empatan con 0,75: gana el menor"
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — Errores por categorías

Un único porcentaje esconde dónde falla el método. Con la tabla `TABLA_ERR` (diez reseñas, con la categoría `A` = sin palabras del léxico, `B` = con negación y `C` = polaridad clara), completa:

- `tasa_error_por_categoria(tabla)`: una `Series` con la **tasa de error** (fracción de filas con `etiqueta != pred`) de cada categoría;
- `matriz_confusion(tabla)`: una tabla que cruza `etiqueta` (filas) y `pred` (columnas) con `pd.crosstab`.

Calcula a mano: `A` falla 2 de 4 (0,50), `B` falla 1 de 2 (0,50) y `C` falla 1 de 4 (0,25).

In [ ]:
TABLA_ERR = pd.DataFrame({
    "categoria": ["A", "A", "A", "A", "B", "B", "C", "C", "C", "C"],
    "etiqueta": ["negativo", "positivo", "neutro", "neutro", "negativo", "positivo", "positivo", "positivo", "negativo", "neutro"],
    "pred": ["neutro", "neutro", "neutro", "neutro", "negativo", "negativo", "positivo", "positivo", "negativo", "positivo"],
})

def tasa_error_por_categoria(tabla):
    # TODO: agrupa por categoría y calcula la media de (etiqueta != pred)
    return None

def matriz_confusion(tabla):
    # TODO: pd.crosstab de etiqueta (filas) y pred (columnas)
    return None

# Comprobación automática
tasas = tasa_error_por_categoria(TABLA_ERR)
matriz = matriz_confusion(TABLA_ERR)
assert tasas is not None and matriz is not None, "Completa las líneas marcadas con TODO"
assert dict(tasas.round(2)) == {"A": 0.5, "B": 0.5, "C": 0.25}, "las tasas de error deben ser A 0,50, B 0,50 y C 0,25"
assert int(matriz.to_numpy().sum()) == 10 and int(matriz.loc["neutro", "neutro"]) == 2, "la matriz debe sumar 10 y tener 2 neutros bien clasificados"
assert int(np.trace(matriz.reindex(index=CLASES, columns=CLASES, fill_value=0).to_numpy())) == 6, "hay 6 aciertos en la diagonal"
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — El clasificador supervisado en la misma partición

Con las reseñas reales de `datos/` (elige `IDIOMA`), completa:

- `particion(df)`: una **única** partición 70/30 estratificada por `etiqueta` con `random_state=42` (devuelve `entren, prueba`);
- `entrenar_y_medir(entren, prueba)`: entrena una `Pipeline` de `TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)` y `LogisticRegression(max_iter=1000)` con el texto **bruto** (`texto`) del entrenamiento, y devuelve `(modelo, exactitud en prueba)`.

Fíjate en el orden: el vectorizador se ajusta solo con el entrenamiento (clase 4). Con 300 reseñas la partición tiene 210 y 90 filas.

In [ ]:
IDIOMA = "spa"            # o "ita"
df_res = pd.read_csv(cargar_datos(ARCHIVO_RESENAS[IDIOMA]))

def particion(df):
    entren, prueba = None, None      # TODO: train_test_split estratificado 70/30 con random_state=42
    return entren, prueba

def entrenar_y_medir(entren, prueba):
    modelo = None                    # TODO: Pipeline TF-IDF (1-2 gramas, sublinear_tf) + regresión logística
    # TODO: ajusta el modelo con entren["texto"] y entren["etiqueta"]
    acierto = None                   # TODO: exactitud sobre prueba["texto"]
    return modelo, acierto

entren, prueba = particion(df_res)
modelo, acierto_lr = entrenar_y_medir(entren, prueba)

# Comprobación automática
assert entren is not None and prueba is not None and modelo is not None and acierto_lr is not None, "Completa las líneas marcadas con TODO"
assert len(entren) == 210 and len(prueba) == 90, "la partición 70/30 de 300 reseñas tiene 210 y 90 filas"
assert set(entren["id"]).isdisjoint(set(prueba["id"])), "entrenamiento y prueba no deben compartir reseñas"
assert prueba["etiqueta"].value_counts().to_dict() == {"negativo": 30, "neutro": 30, "positivo": 30}, "la partición debe estar estratificada: 30 reseñas por clase en la prueba"
assert isinstance(modelo.steps[0][1], TfidfVectorizer) and isinstance(modelo.steps[-1][1], LogisticRegression), "la Pipeline debe ser TfidfVectorizer + LogisticRegression"
assert modelo.steps[0][1].ngram_range == (1, 2) and modelo.steps[0][1].sublinear_tf, "usa ngram_range=(1, 2) y sublinear_tf=True"
assert 0.5 < acierto_lr <= 1.0, "la exactitud debería estar claramente por encima del azar (1/3)"
assert abs(acierto_lr - modelo.score(prueba["texto"], prueba["etiqueta"])) < 1e-12
print(f"Ejercicio 5 correcto ✔ (exactitud en prueba: {acierto_lr:.3f}; cifra ilustrativa)")

---
## Reto final — ¿Ayuda el léxico al modelo supervisado?

Con la partición del ejercicio 5 (`entren`, `prueba`) y el **léxico real del curso** (`LEXICO_REAL`, ya cargado, con `analizar` de la clase), construye un modelo **híbrido**: las columnas de TF-IDF **más una columna extra** con la puntuación del léxico de cada reseña.

1. Escribe `puntuacion_lexico(textos)`: array de `float` con `puntuar(texto, LEXICO_REAL[IDIOMA], IDIOMA)` para cada texto.
2. Escribe `medir_hibrido()`: entrena TF-IDF (1-2 gramas, `sublinear_tf`) con `entren["texto"]`, añade la columna del léxico con `scipy.sparse.hstack` (ajusta el vectorizador **solo** con el entrenamiento), entrena `LogisticRegression(max_iter=1000)` y devuelve un diccionario con las exactitudes en la **misma prueba**: `{"solo_tfidf": ..., "hibrido": ...}`.

No se pide que el híbrido gane: **mide y describe** lo que sale. Con 90 reseñas de prueba, una diferencia de pocos puntos es ruido.

In [ ]:
LEXICO_REAL = {idioma: cargar_lexico(idioma) for idioma in IDIOMAS}

def puntuacion_lexico(textos):
    # TODO: array de floats con la puntuación del léxico de cada texto
    return None

def medir_hibrido():
    # TODO: TF-IDF ajustado con entren + columna del léxico (sp.hstack) + regresión logística; devuelve el diccionario
    return None

resultado = medir_hibrido()

# Comprobación automática
assert resultado is not None, "Completa las líneas marcadas con TODO"
assert set(resultado) == {"solo_tfidf", "hibrido"}, "el diccionario debe tener las claves solo_tfidf e hibrido"
assert all(0.5 < v <= 1.0 for v in resultado.values()), "las dos exactitudes deben estar claramente por encima del azar"
assert abs(resultado["solo_tfidf"] - acierto_lr) < 1e-12, "«solo_tfidf» debe coincidir con la exactitud del ejercicio 5 (misma partición y mismo modelo)"
ejemplo = puntuacion_lexico(prueba["texto"].head(5))
assert len(ejemplo) == 5 and ejemplo[0] == puntuar(prueba["texto"].iloc[0], LEXICO_REAL[IDIOMA], IDIOMA), "puntuacion_lexico debe usar puntuar con el léxico real"
print(f"Reto final correcto ✔ (solo TF-IDF {resultado['solo_tfidf']:.3f} · híbrido {resultado['hibrido']:.3f}; cifras ilustrativas)")

**Tus conclusiones (escribe 3-4 frases):** en el reto, ¿el híbrido mejora, empeora o queda igual que el TF-IDF solo? ¿Cuántas reseñas de prueba de diferencia son? ¿Cambiarías de conclusión con otra partición? Menciona una categoría de error (negación, ironía, sin palabras del léxico) donde crees que el léxico ayudaría o no.

*(Tu respuesta aquí.)*

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
def puntuar_simple(frase, lexico):
    tokens = re.findall(r"\w+", frase.lower())
    total = sum(lexico.get(t, 0) for t in tokens)
    return total

puntos_1 = {idioma: [puntuar_simple(f, LEXICO_MINI[idioma]) for f in FRASES_P[idioma]] for idioma in IDIOMAS}

# Comprobación automática
assert all(p is not None for idioma in IDIOMAS for p in puntos_1[idioma]), "Completa las líneas marcadas con TODO"
for idioma in IDIOMAS:
    assert puntos_1[idioma] == [2, 2, -2, 1, -1, -2, 0], "las puntuaciones deben ser 2, 2, −2, 1, −1, −2, 0"
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
def puntuar_neg(frase, lexico, negadores, ventana):
    tokens = re.findall(r"\w+", frase.lower())
    total = 0
    for i, t in enumerate(tokens):
        if t in lexico:
            valor = lexico[t]
            anteriores = tokens[max(0, i - ventana):i]
            if any(a in negadores for a in anteriores):
                valor = -valor
            total += valor
    return total

FRASE_LEJOS = {"spa": "no es un servicio bueno", "ita": "non è un servizio buono"}   # dato-es
res_2 = {idioma: ([puntuar_neg(f, LEXICO_MINI[idioma], NEGADORES_P[idioma], 2) for f in FRASES_P[idioma]],
                  puntuar_neg(FRASE_LEJOS[idioma], LEXICO_MINI[idioma], NEGADORES_P[idioma], 2),
                  puntuar_neg(FRASE_LEJOS[idioma], LEXICO_MINI[idioma], NEGADORES_P[idioma], 4)) for idioma in IDIOMAS}

# Comprobación automática
for idioma in IDIOMAS:
    lista, cerca, lejos = res_2[idioma]
    assert lista == [2, 2, -2, -1, 1, -2, 0], "con ventana=2 las puntuaciones deben ser 2, 2, −2, −1, 1, −2, 0"
    assert cerca == 1 and lejos == -1, "con ventana=2 la frase lejana vale +1 y con ventana=4 vale −1"
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
PUNTOS = [2, -1, 0, 1, -2, 0, 3, -1]
ETIQUETAS = ["positivo", "negativo", "neutro", "neutro", "negativo", "neutro", "positivo", "positivo"]

def clasificar(p, umbral):
    if p > umbral:
        return "positivo"
    if p < -umbral:
        return "negativo"
    return "neutro"

def exactitud(y, pred):
    return sum(a == b for a, b in zip(y, pred)) / len(y)

def elegir_umbral(puntos, y, candidatos):
    mejor, mejor_acierto = None, -1
    for u in sorted(candidatos):
        acierto = exactitud(y, [clasificar(p, u) for p in puntos])
        if acierto > mejor_acierto:
            mejor, mejor_acierto = u, acierto
    return mejor

# Comprobación automática
assert clasificar(0, 0) is not None and exactitud(["a"], ["a"]) is not None and elegir_umbral(PUNTOS, ETIQUETAS, (0, 1)) is not None, "Completa las líneas marcadas con TODO"
assert [clasificar(p, 0) for p in (2, -1, 0)] == ["positivo", "negativo", "neutro"], "con umbral 0: 2 → positivo, −1 → negativo, 0 → neutro"
assert clasificar(1, 1) == "neutro" and clasificar(-1, 1) == "neutro", "el umbral es estricto: p = umbral es neutro"
assert exactitud(ETIQUETAS, [clasificar(p, 0) for p in PUNTOS]) == 0.75
assert exactitud(ETIQUETAS, [clasificar(p, 1) for p in PUNTOS]) == 0.75
assert exactitud(ETIQUETAS, [clasificar(p, 2) for p in PUNTOS]) == 0.5
assert elegir_umbral(PUNTOS, ETIQUETAS, (2, 1, 0)) == 0, "0 y 1 empatan con 0,75: gana el menor"
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
TABLA_ERR = pd.DataFrame({
    "categoria": ["A", "A", "A", "A", "B", "B", "C", "C", "C", "C"],
    "etiqueta": ["negativo", "positivo", "neutro", "neutro", "negativo", "positivo", "positivo", "positivo", "negativo", "neutro"],
    "pred": ["neutro", "neutro", "neutro", "neutro", "negativo", "negativo", "positivo", "positivo", "negativo", "positivo"],
})

def tasa_error_por_categoria(tabla):
    return (tabla["etiqueta"] != tabla["pred"]).groupby(tabla["categoria"]).mean()

def matriz_confusion(tabla):
    return pd.crosstab(tabla["etiqueta"], tabla["pred"])

# Comprobación automática
tasas = tasa_error_por_categoria(TABLA_ERR)
matriz = matriz_confusion(TABLA_ERR)
assert tasas is not None and matriz is not None, "Completa las líneas marcadas con TODO"
assert dict(tasas.round(2)) == {"A": 0.5, "B": 0.5, "C": 0.25}, "las tasas de error deben ser A 0,50, B 0,50 y C 0,25"
assert int(matriz.to_numpy().sum()) == 10 and int(matriz.loc["neutro", "neutro"]) == 2, "la matriz debe sumar 10 y tener 2 neutros bien clasificados"
assert int(np.trace(matriz.reindex(index=CLASES, columns=CLASES, fill_value=0).to_numpy())) == 6, "hay 6 aciertos en la diagonal"
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
IDIOMA = "spa"
df_res = pd.read_csv(cargar_datos(ARCHIVO_RESENAS[IDIOMA]))

def particion(df):
    entren, prueba = train_test_split(df, test_size=0.3, stratify=df["etiqueta"], random_state=42)
    return entren, prueba

def entrenar_y_medir(entren, prueba):
    modelo = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True), LogisticRegression(max_iter=1000))
    modelo.fit(entren["texto"], entren["etiqueta"])
    acierto = accuracy_score(prueba["etiqueta"], modelo.predict(prueba["texto"]))
    return modelo, acierto

entren, prueba = particion(df_res)
modelo, acierto_lr = entrenar_y_medir(entren, prueba)

# Comprobación automática
assert entren is not None and prueba is not None and modelo is not None and acierto_lr is not None, "Completa las líneas marcadas con TODO"
assert len(entren) == 210 and len(prueba) == 90, "la partición 70/30 de 300 reseñas tiene 210 y 90 filas"
assert set(entren["id"]).isdisjoint(set(prueba["id"])), "entrenamiento y prueba no deben compartir reseñas"
assert prueba["etiqueta"].value_counts().to_dict() == {"negativo": 30, "neutro": 30, "positivo": 30}, "la partición debe estar estratificada: 30 reseñas por clase en la prueba"
assert isinstance(modelo.steps[0][1], TfidfVectorizer) and isinstance(modelo.steps[-1][1], LogisticRegression), "la Pipeline debe ser TfidfVectorizer + LogisticRegression"
assert modelo.steps[0][1].ngram_range == (1, 2) and modelo.steps[0][1].sublinear_tf, "usa ngram_range=(1, 2) y sublinear_tf=True"
assert 0.5 < acierto_lr <= 1.0, "la exactitud debería estar claramente por encima del azar (1/3)"
assert abs(acierto_lr - modelo.score(prueba["texto"], prueba["etiqueta"])) < 1e-12
print(f"Ejercicio 5 correcto ✔ (exactitud en prueba: {acierto_lr:.3f}; cifra ilustrativa)")

### Solución del reto final

In [ ]:
LEXICO_REAL = {idioma: cargar_lexico(idioma) for idioma in IDIOMAS}

def puntuacion_lexico(textos):
    return np.array([puntuar(t, LEXICO_REAL[IDIOMA], IDIOMA) for t in textos], dtype=float)

def medir_hibrido():
    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    X_tr = vec.fit_transform(entren["texto"])
    X_te = vec.transform(prueba["texto"])
    solo = LogisticRegression(max_iter=1000).fit(X_tr, entren["etiqueta"])
    H_tr = sp.hstack([X_tr, sp.csr_matrix(puntuacion_lexico(entren["texto"]).reshape(-1, 1))]).tocsr()
    H_te = sp.hstack([X_te, sp.csr_matrix(puntuacion_lexico(prueba["texto"]).reshape(-1, 1))]).tocsr()
    hibrido = LogisticRegression(max_iter=1000).fit(H_tr, entren["etiqueta"])
    return {"solo_tfidf": accuracy_score(prueba["etiqueta"], solo.predict(X_te)),
            "hibrido": accuracy_score(prueba["etiqueta"], hibrido.predict(H_te))}

resultado = medir_hibrido()

# Comprobación automática
assert resultado is not None, "Completa las líneas marcadas con TODO"
assert set(resultado) == {"solo_tfidf", "hibrido"}, "el diccionario debe tener las claves solo_tfidf e hibrido"
assert all(0.5 < v <= 1.0 for v in resultado.values()), "las dos exactitudes deben estar claramente por encima del azar"
assert abs(resultado["solo_tfidf"] - acierto_lr) < 1e-12, "«solo_tfidf» debe coincidir con la exactitud del ejercicio 5 (misma partición y mismo modelo)"
ejemplo = puntuacion_lexico(prueba["texto"].head(5))
assert len(ejemplo) == 5 and ejemplo[0] == puntuar(prueba["texto"].iloc[0], LEXICO_REAL[IDIOMA], IDIOMA), "puntuacion_lexico debe usar puntuar con el léxico real"
print(f"Reto final correcto ✔ (solo TF-IDF {resultado['solo_tfidf']:.3f} · híbrido {resultado['hibrido']:.3f}; cifras ilustrativas)")

---
## Autoevaluación de la clase

Responde con sinceridad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto de la clase no te ha quedado claro (léxico de polaridad, ventana de negación, umbral, comparación en la misma partición, categorías de error, ironía)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*